# Current one-arm WDM profiling

This notebook replaces the legacy `profile_one_arm_wdm_pipeline.ipynb` workflow with the current prepared-plan architecture.

It is deliberately narrower than `getting_started.ipynb` and the scheduling sweep notebook. Its purpose is to locate the remaining cost **inside A1** after the current production choices have already been fixed:

- CPU backend;
- `assembly_backend="lagfirst_chunked_lagblock"`;
- `assembly_precision="complex64"`;
- `row_chunk_size=4096`;
- `lag_block_size=32`;
- device-materialized one-arm execution as the production route.

The notebook answers five diagnostic questions:

1. How much of A1 is spent in the two DWTs, six-link modulation/materialization, and the variable shift?
2. Does the `VariableShiftBatchPlan.apply_device` wrapper add measurable overhead above the compiled kernel?
3. Does the shift scale linearly from one to six links?
4. Is batched execution better or worse than six separate one-link calls?
5. Are any individual links anomalously expensive?

An optional JAX profiler trace is included at the end. No production source files are modified by this notebook.


In [1]:
# --- 1. CPU-first imports and implementation fingerprint ---

import gc
import inspect
import os
import platform
import sys
import time
from datetime import datetime
from pathlib import Path

# This must be set before importing JAX, FEW, WDM, or WDM_LISAresponse.
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    _normalise_one_arm_execution_mode,
)

from WDM.code.time_delay_filters._time_shift_jax import (
    assemble_shift_target_batch_chunked_lagblock_jax,
)

import WDM
import WDM_LISAresponse
import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM_LISAresponse.code.lisa_response.response_with_plan as response_with_plan_module


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("one_arm_plan:", one_arm_plan_module.__file__)
print("response_with_plan:", response_with_plan_module.__file__)
print("OneArmWDMPlan.apply_device:", inspect.signature(OneArmWDMPlan.apply_device))
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu", (
    "Restart the kernel and run this cell first. JAX must be forced to CPU "
    "before any JAX-related import."
)
assert all(device.platform == "cpu" for device in jax.devices())
assert _normalise_one_arm_execution_mode("materialized") == "materialized"
assert "execution_mode" in inspect.signature(OneArmWDMPlan.apply_device).parameters


Timestamp: 2026-07-16T16:16:08
Python executable: /data/tk593/conda-envs/lisatools/bin/python
Python: 3.12.12
Platform: Linux-5.14.0-687.24.1.el9_8.x86_64-x86_64-with-glibc2.34
JAX: 0.9.2
JAX backend: cpu
JAX devices: [CpuDevice(id=0)]
WDM: /data/tk593/LISA_Response/LISAresponse/WDM_GW_wavelets/src/WDM/__init__.py
WDM_LISAresponse: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
one_arm_plan: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/code/lisa_response/one_arm_plan.py
response_with_plan: /data/tk593/LISA_Response/LISAresponse/WDM_LISAresponse/src/WDM_LISAresponse/code/lisa_response/response_with_plan.py
OneArmWDMPlan.apply_device: (self, w_t_plus, w_t_cross, *, execution_mode: 'str' = 'materialized', return_profile: 'bool' = False, cache_device_plan: 'bool' = True)
RSS: 0.81 GiB


In [2]:
# --- 2. Scientific and profiling configuration ---

# Full-year production-like case.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

# Current confirmed CPU schedule.
ROW_CHUNK_SIZE = 4096
LAG_BLOCK_SIZE = 32
ASSEMBLY_PRECISION = "complex64"
ASSEMBLY_BACKEND = "lagfirst_chunked_lagblock"

# Windowing.
alpha = 0.005
tukey_factor = 4

# Orbit and sky position.
ORBIT_FILE = "esa-trailing-orbits.h5"
theta = np.pi / 2
phi = 0.0

# EMRI parameters.
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

# Timing controls.
PROFILE_WARMUPS = 2
PROFILE_REPEATS = 15
SCALING_REPEATS = 9
LINK_REPEATS = 9

# Optional trace. Leave False for ordinary profiling.
RUN_JAX_TRACE = False
TRACE_REPEATS = 5
TRACE_DIRECTORY = Path("jax_profile_one_arm_shift")

# Save compact CSV summaries.
SAVE_RESULTS = False
RESULTS_PREFIX = "profile_one_arm_wdm_current"

SHIFT_KWARGS = {
    "assembly_backend": ASSEMBLY_BACKEND,
    "assembly_precision": ASSEMBLY_PRECISION,
    "row_chunk_size": ROW_CHUNK_SIZE,
    "lag_block_size": LAG_BLOCK_SIZE,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
}

print("Full-year samples:", N)
print("Expected WDM Nt:", N // (2 * Nf))
print("L_trunc:", L_TRUNC, "| lags:", 2 * L_TRUNC + 1)
print("Shift configuration:", SHIFT_KWARGS)
print("Warmups/repeats:", PROFILE_WARMUPS, PROFILE_REPEATS)


Full-year samples: 3200000
Expected WDM Nt: 16000
L_trunc: 25 | lags: 51
Shift configuration: {'assembly_backend': 'lagfirst_chunked_lagblock', 'assembly_precision': 'complex64', 'row_chunk_size': 4096, 'lag_block_size': 32, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear', 'use_jax': True}
Warmups/repeats: 2 15


In [3]:
# --- 3. Timing, synchronization, validation, and reporting helpers ---

def block_result(value):
    """Synchronize all JAX leaves before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def summarize_timings(values):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size == 0:
        raise ValueError("At least one one-dimensional timing sample is required.")

    q25, median, q75 = np.quantile(values, [0.25, 0.50, 0.75])

    return {
        "median_s": float(median),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "mean_s": float(np.mean(values)),
        "std_s": float(np.std(values)),
        "times_s": [float(value) for value in values],
    }


def compare_device_batches(candidate, reference):
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    delta = candidate - reference
    max_abs = jnp.max(jnp.abs(delta))
    reference_norm = jnp.linalg.norm(reference)
    rel_l2 = jnp.where(
        reference_norm > 0,
        jnp.linalg.norm(delta) / reference_norm,
        jnp.nan,
    )

    block_result((max_abs, rel_l2))

    return {
        "max_abs": float(max_abs),
        "rel_l2": float(rel_l2),
    }


def rotating_benchmark(routes, *, warmups, repeats):
    """Benchmark routes with fair ordering and explicit synchronization."""

    route_names = list(routes)

    for _ in range(max(0, int(warmups))):
        for name in route_names:
            result = routes[name]()
            block_result(result)
            del result

    timings = {
        name: []
        for name in route_names
    }

    for repeat_index in range(max(1, int(repeats))):
        if len(route_names) == 2:
            # True alternating order for pairwise comparisons.
            if repeat_index % 2 == 0:
                order = list(route_names)
            else:
                order = list(reversed(route_names))
        else:
            # Rotate multi-route tests without the reversal that can
            # cancel the rotation for a two-item list.
            rotation = repeat_index % len(route_names)
            order = (
                route_names[rotation:]
                + route_names[:rotation]
            )

        print(
            f"Round {repeat_index + 1:02d}/{repeats}: "
            f"order={order}"
        )

        for name in order:
            started = time.perf_counter()

            result = routes[name]()
            block_result(result)

            timings[name].append(
                time.perf_counter() - started
            )

            del result

    return {
        name: summarize_timings(values)
        for name, values in timings.items()
    }

def print_summary(name, summary):
    print(
        f"{name:>28}: "
        f"median={summary['median_s']:.3f} s | "
        f"IQR={summary['iqr_s']:.3f} s | "
        f"[q25,q75]=[{summary['q25_s']:.3f}, "
        f"{summary['q75_s']:.3f}]"
    )


def show_rows(rows, *, sort_by=None):
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by is not None and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, kind="stable").reset_index(drop=True)
        display(frame)
        return frame

    rows_out = list(rows)
    if sort_by is not None:
        rows_out = sorted(rows_out, key=lambda row: row.get(sort_by, np.inf))
    for row in rows_out:
        print(row)
    return rows_out


def save_rows(rows, filename):
    if not SAVE_RESULTS:
        return
    if pd is None:
        print("pandas unavailable; skipping:", filename)
        return
    pd.DataFrame(rows).to_csv(filename, index=False)
    print("Saved:", filename)


## Build the fixed waveform and current one-arm plan

The waveform, WDM grid, DWT coefficients, and geometry-dependent plan are built once. Plan build and waveform generation are setup costs; repeated profiling starts after these objects exist.


In [4]:
# --- 4. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

print("Orbit samples:", orbit_data.size)
print("Starting waveform generation...")


Orbit samples: 1721
Starting waveform generation...


In [5]:
# --- 5. Generate and pad the waveform once ---

started = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model="emri",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function="tukey",
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

waveform_generation_s = time.perf_counter() - started

print("Waveform generation_s:", f"{waveform_generation_s:.3f}")
print("times_pad:", times_pad.shape, times_pad.dtype)
print("hplus_td:", hplus_td.shape, hplus_td.dtype)
print("hcross_td:", hcross_td.shape, hcross_td.dtype)
print("WDM Nt x Nf:", pad_info["wdm_Nt"], "x", Nf)
print("RSS:", f"{rss_gib():.2f} GiB")


Waveform generation_s: 8.853
times_pad: (3264000,) float64
hplus_td: (3264000,) float64
hcross_td: (3264000,) float64
WDM Nt x Nf: 32640 x 100
RSS: 6.52 GiB


In [6]:
# --- 6. Build WDM grid, perform the two DWTs, and build the current plan ---

started = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - started

started = time.perf_counter()
wave_context = _build_wdm_dT_context_from_wdm(
    times_pad,
    hplus_td,
    hcross_td,
    grid_context["wdm"],
)
block_result((wave_context["w_t_plus"], wave_context["w_t_cross"]))
initial_two_dwt_s = time.perf_counter() - started

started = time.perf_counter()
one_arm_plan = OneArmWDMPlan.build_from_context(
    grid_context,
    orbits_trailing,
    theta,
    phi,
    L_trunc=L_TRUNC,
    variable_shift_kwargs=SHIFT_KWARGS,
    use_jax=True,
)
one_arm_plan_build_s = time.perf_counter() - started

shift_plan = one_arm_plan.shift_plan
wdm = grid_context["wdm"]

assert shift_plan.resolved_assembly_backend == ASSEMBLY_BACKEND
assert shift_plan.resolved_assembly_precision == ASSEMBLY_PRECISION
assert not hasattr(shift_plan, "apply_device_interior_fast")
assert not hasattr(shift_plan, "apply_device_interior_edge_strip")

w_plus_host = wave_context["w_t_plus"]
w_cross_host = wave_context["w_t_cross"]

shift_dtype = (
    jnp.complex64
    if shift_plan.resolved_assembly_precision == "complex64"
    else jnp.complex128
)

w_plus_device = jnp.asarray(w_plus_host, dtype=shift_dtype)
w_cross_device = jnp.asarray(w_cross_host, dtype=shift_dtype)
block_result((w_plus_device, w_cross_device))

print("WDM grid build_s:", f"{grid_build_s:.3f}")
print("Initial two DWTs_s:", f"{initial_two_dwt_s:.3f}")
print("One-arm plan build_s:", f"{one_arm_plan_build_s:.3f}")
print("WDM shape:", (wdm.Nt, wdm.Nf))
print("WDM dT/dF:", wdm.dT, wdm.dF)
print("Plan jobs:", shift_plan.num_jobs)
print("Plan offset:", shift_plan.offset)
print("Backend:", shift_plan.resolved_assembly_backend)
print("Precision:", shift_plan.resolved_assembly_precision)
print("Persistent plan memory_MiB:", shift_plan.device_plan_memory_bytes / 1024**2)
print("RSS:", f"{rss_gib():.2f} GiB")


WDM grid build_s: 2.955
Initial two DWTs_s: 0.783
One-arm plan build_s: 0.994
WDM shape: (32640, 100)
WDM dT/dF: 1000.0 0.0005
Plan jobs: 6
Plan offset: 25
Backend: lagfirst_chunked_lagblock
Precision: complex64
Persistent plan memory_MiB: 331.2015609741211
RSS: 8.23 GiB


## A1 decomposition

This section times the current production path and its main constituents:

- `two_DWTs`: rebuild only the waveform WDM context;
- `materialization_only`: form the six unshifted link coefficient arrays;
- `shift_only`: apply the prepared six-job variable shift;
- `complete_materialized`: current production one-arm route from device-resident polarizations;
- `fused_low_memory`: optional low-memory route.

The routes are warmed first and then timed in rotating order with explicit JAX synchronization.


In [7]:
# --- 7. Prepare the six-link intermediate and define A1 routes ---

unshifted_host = one_arm_plan.unshifted_coefficients(
    w_plus_host,
    w_cross_host,
)
unshifted_device = jnp.asarray(unshifted_host, dtype=shift_dtype)
block_result(unshifted_device)

print("Unshifted six-link shape:", unshifted_device.shape)
print("Unshifted six-link dtype:", unshifted_device.dtype)
print("Unshifted six-link memory_MiB:", unshifted_device.nbytes / 1024**2)


def run_two_dwts():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        grid_context["wdm"],
    )


def run_materialization_only():
    result = one_arm_plan.unshifted_coefficients(
        w_plus_device,
        w_cross_device,
    )
    return jnp.asarray(result, dtype=shift_dtype)


def run_shift_only():
    return shift_plan.apply_device(unshifted_device)


def run_complete_materialized():
    return one_arm_plan.apply_batch_device_materialized(
        w_plus_device,
        w_cross_device,
    )


def run_fused_low_memory():
    return one_arm_plan.apply_batch_device(
        w_plus_device,
        w_cross_device,
    )


# Validate all complete/shift routes before timing.
reference = run_complete_materialized()
block_result(reference)

for name, function in {
    "shift_only": run_shift_only,
    "fused_low_memory": run_fused_low_memory,
}.items():
    candidate = function()
    block_result(candidate)
    comparison = compare_device_batches(candidate, reference)
    np.testing.assert_allclose(
        np.asarray(candidate),
        np.asarray(reference),
        rtol=2e-6,
        atol=2e-7,
    )
    print(
        f"Validated {name}: "
        f"max_abs={comparison['max_abs']:.3e}, "
        f"rel_l2={comparison['rel_l2']:.3e}"
    )
    del candidate

del reference
gc.collect()


Unshifted six-link shape: (6, 32640, 100)
Unshifted six-link dtype: complex64
Unshifted six-link memory_MiB: 149.4140625
Validated shift_only: max_abs=1.084e-18, rel_l2=7.790e-08
Validated fused_low_memory: max_abs=1.084e-18, rel_l2=8.385e-08


0

In [8]:
# --- 8. Rotating-order A1 benchmark ---

a1_routes = {
    "two_DWTs": run_two_dwts,
    "materialization_only": run_materialization_only,
    "shift_only": run_shift_only,
    "complete_materialized": run_complete_materialized,
    "fused_low_memory": run_fused_low_memory,
}

a1_summaries = rotating_benchmark(
    a1_routes,
    warmups=PROFILE_WARMUPS,
    repeats=PROFILE_REPEATS,
)

print()
print("=" * 80)
print("A1 decomposition")
print("=" * 80)

for name in a1_routes:
    print_summary(name, a1_summaries[name])

dwt_median = a1_summaries["two_DWTs"]["median_s"]
materialization_median = a1_summaries["materialization_only"]["median_s"]
shift_median = a1_summaries["shift_only"]["median_s"]
complete_median = a1_summaries["complete_materialized"]["median_s"]
fused_median = a1_summaries["fused_low_memory"]["median_s"]

print()
print("Derived diagnostics")
print("-" * 80)
print(
    "shift share of complete materialized route:",
    f"{100.0 * shift_median / complete_median:.1f}%",
)
print(
    "complete minus shift:",
    f"{complete_median - shift_median:+.3f} s",
)
print(
    "standalone materialization:",
    f"{materialization_median:.3f} s",
)
print(
    "fused penalty:",
    f"{fused_median - complete_median:+.3f} s "
    f"({100.0 * (fused_median / complete_median - 1.0):+.1f}%)",
)
print(
    "two DWTs + complete materialized:",
    f"{dwt_median + complete_median:.3f} s",
)

a1_rows = []
for name, summary in a1_summaries.items():
    row = {"route": name}
    row.update(summary)
    a1_rows.append(row)

save_rows(a1_rows, f"{RESULTS_PREFIX}_a1_decomposition.csv")


Round 01/15: order=['two_DWTs', 'materialization_only', 'shift_only', 'complete_materialized', 'fused_low_memory']
Round 02/15: order=['materialization_only', 'shift_only', 'complete_materialized', 'fused_low_memory', 'two_DWTs']
Round 03/15: order=['shift_only', 'complete_materialized', 'fused_low_memory', 'two_DWTs', 'materialization_only']
Round 04/15: order=['complete_materialized', 'fused_low_memory', 'two_DWTs', 'materialization_only', 'shift_only']
Round 05/15: order=['fused_low_memory', 'two_DWTs', 'materialization_only', 'shift_only', 'complete_materialized']
Round 06/15: order=['two_DWTs', 'materialization_only', 'shift_only', 'complete_materialized', 'fused_low_memory']
Round 07/15: order=['materialization_only', 'shift_only', 'complete_materialized', 'fused_low_memory', 'two_DWTs']
Round 08/15: order=['shift_only', 'complete_materialized', 'fused_low_memory', 'two_DWTs', 'materialization_only']
Round 09/15: order=['complete_materialized', 'fused_low_memory', 'two_DWTs', 'ma

## Wrapper versus compiled kernel

The production `shift_plan.apply_device` wrapper performs validation, device-cache lookup, chunk slicing, and dispatch. This test compares it with a direct call to the compiled lag-blocked kernel using the already cached device-plan arrays.

A negligible difference means the remaining cost is inside the compiled numerical kernel rather than Python or plan dispatch.


In [9]:
# --- 9. Wrapper-versus-kernel diagnostic ---

device_plan = shift_plan._device_plan_arrays()
block_result(device_plan)


def run_direct_kernel(start=0, stop=None):
    if stop is None:
        stop = shift_plan.num_jobs

    return assemble_shift_target_batch_chunked_lagblock_jax(
        unshifted_device[start:stop],
        device_plan["delays"][start:stop],
        device_plan["ell_all"],
        shift_plan.offset,
        device_plan["Tl_all"][start:stop],
        device_plan["Tp_all"][start:stop],
        device_plan["Cnm"],
        float(shift_plan.wdm.dF),
        row_chunk_size=ROW_CHUNK_SIZE,
        lag_block_size=LAG_BLOCK_SIZE,
        precision=ASSEMBLY_PRECISION,
        return_device=True,
    )


direct_reference = run_direct_kernel()
wrapper_reference = run_shift_only()
block_result((direct_reference, wrapper_reference))

wrapper_kernel_comparison = compare_device_batches(
    direct_reference,
    wrapper_reference,
)
np.testing.assert_allclose(
    np.asarray(direct_reference),
    np.asarray(wrapper_reference),
    rtol=2e-6,
    atol=2e-7,
)

wrapper_kernel_summaries = rotating_benchmark(
    {
        "shift_plan_apply_device": run_shift_only,
        "direct_compiled_kernel": run_direct_kernel,
    },
    warmups=PROFILE_WARMUPS,
    repeats=PROFILE_REPEATS,
)

print()
print("=" * 80)
print("Wrapper versus direct compiled kernel")
print("=" * 80)
for name, summary in wrapper_kernel_summaries.items():
    print_summary(name, summary)

wrapper_median = wrapper_kernel_summaries[
    "shift_plan_apply_device"
]["median_s"]
kernel_median = wrapper_kernel_summaries[
    "direct_compiled_kernel"
]["median_s"]

print()
print(
    "wrapper overhead estimate:",
    f"{wrapper_median - kernel_median:+.4f} s "
    f"({100.0 * (wrapper_median / kernel_median - 1.0):+.2f}%)",
)
print("max_abs:", f"{wrapper_kernel_comparison['max_abs']:.6e}")
print("rel_l2:", f"{wrapper_kernel_comparison['rel_l2']:.6e}")

wrapper_rows = []
for name, summary in wrapper_kernel_summaries.items():
    row = {"route": name}
    row.update(summary)
    wrapper_rows.append(row)

save_rows(wrapper_rows, f"{RESULTS_PREFIX}_wrapper_vs_kernel.csv")


Round 01/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 02/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 03/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 04/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 05/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 06/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 07/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 08/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 09/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 10/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 11/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 12/15: order=['direct_compiled_kernel', 'shift_plan_apply_device']
Round 13/15: order=['shift_plan_apply_device', 'direct_compiled_kernel']
Round 14/15: order=['direct_compiled_kernel', 'shif

## Shift scaling with link count

The current batched kernel internally applies the same prepared operation to six jobs. Timing prefixes of size \(B=1,\ldots,6\) reveals whether cost is approximately linear in the number of links or dominated by a fixed batch-level overhead.


In [10]:
# --- 10. Batch scaling from one to six links ---

BATCH_SIZES = tuple(range(1, shift_plan.num_jobs + 1))

# Compile and warm every distinct batch shape.
for batch_size in BATCH_SIZES:
    for _ in range(PROFILE_WARMUPS):
        result = run_direct_kernel(0, batch_size)
        block_result(result)
        del result

batch_times = {batch_size: [] for batch_size in BATCH_SIZES}

for repeat_index in range(SCALING_REPEATS):
    rotation = repeat_index % len(BATCH_SIZES)
    order = list(BATCH_SIZES[rotation:] + BATCH_SIZES[:rotation])

    if repeat_index % 2 == 1:
        order = list(reversed(order))

    print(
        f"Scaling round {repeat_index + 1:02d}/{SCALING_REPEATS}: "
        f"order={order}"
    )

    for batch_size in order:
        started = time.perf_counter()
        result = run_direct_kernel(0, batch_size)
        block_result(result)
        batch_times[batch_size].append(time.perf_counter() - started)
        del result

batch_summaries = {
    batch_size: summarize_timings(values)
    for batch_size, values in batch_times.items()
}

batch_sizes_array = np.asarray(BATCH_SIZES, dtype=float)
batch_medians = np.asarray(
    [batch_summaries[size]["median_s"] for size in BATCH_SIZES],
    dtype=float,
)

slope, intercept = np.polyfit(batch_sizes_array, batch_medians, deg=1)
prediction = slope * batch_sizes_array + intercept
ss_residual = float(np.sum((batch_medians - prediction) ** 2))
ss_total = float(np.sum((batch_medians - np.mean(batch_medians)) ** 2))
linear_r2 = 1.0 - ss_residual / ss_total if ss_total > 0.0 else 1.0

scaling_rows = []

print()
print("=" * 80)
print("Production shift scaling with link count")
print("=" * 80)
print(f"{'links':>5} {'median_s':>10} {'IQR_s':>10} {'s/link':>10}")

for batch_size in BATCH_SIZES:
    summary = batch_summaries[batch_size]
    seconds_per_link = summary["median_s"] / batch_size

    print(
        f"{batch_size:>5d} "
        f"{summary['median_s']:>10.3f} "
        f"{summary['iqr_s']:>10.3f} "
        f"{seconds_per_link:>10.3f}"
    )

    row = {
        "links": batch_size,
        "seconds_per_link": seconds_per_link,
    }
    row.update(summary)
    scaling_rows.append(row)

print()
print("Linear slope per additional link:", f"{slope:.4f} s")
print("Linear intercept:", f"{intercept:.4f} s")
print("Linear-fit R^2:", f"{linear_r2:.4f}")

save_rows(scaling_rows, f"{RESULTS_PREFIX}_batch_scaling.csv")


Scaling round 01/9: order=[1, 2, 3, 4, 5, 6]
Scaling round 02/9: order=[1, 6, 5, 4, 3, 2]
Scaling round 03/9: order=[3, 4, 5, 6, 1, 2]
Scaling round 04/9: order=[3, 2, 1, 6, 5, 4]
Scaling round 05/9: order=[5, 6, 1, 2, 3, 4]
Scaling round 06/9: order=[5, 4, 3, 2, 1, 6]
Scaling round 07/9: order=[1, 2, 3, 4, 5, 6]
Scaling round 08/9: order=[1, 6, 5, 4, 3, 2]
Scaling round 09/9: order=[3, 4, 5, 6, 1, 2]

Production shift scaling with link count
links   median_s      IQR_s     s/link
    1      0.097      0.002      0.097
    2      0.202      0.005      0.101
    3      0.301      0.007      0.100
    4      0.399      0.003      0.100
    5      0.499      0.012      0.100
    6      0.570      0.011      0.095

Linear slope per additional link: 0.0958 s
Linear intercept: 0.0093 s
Linear-fit R^2: 0.9972


## Batched six-link execution versus six one-link calls

This separates a possible batching problem from the cost of the individual shifts. The sequential route invokes the already compiled \(B=1\) kernel six times and concatenates the results.

- Similar timings imply that the six-link batch wrapper is not the problem.
- A substantially slower batched route would point to the batched `fori_loop` implementation or its compiled layout.


In [11]:
# --- 11. Six-link batch versus six one-link calls ---

def run_six_as_one_batch():
    return run_direct_kernel(0, shift_plan.num_jobs)


def run_six_sequential_single_links():
    outputs = [
        run_direct_kernel(link_index, link_index + 1)
        for link_index in range(shift_plan.num_jobs)
    ]
    return jnp.concatenate(outputs, axis=0)


batch_reference = run_six_as_one_batch()
sequential_reference = run_six_sequential_single_links()
block_result((batch_reference, sequential_reference))

batch_sequential_comparison = compare_device_batches(
    sequential_reference,
    batch_reference,
)
np.testing.assert_allclose(
    np.asarray(sequential_reference),
    np.asarray(batch_reference),
    rtol=2e-6,
    atol=2e-7,
)

batch_sequential_summaries = rotating_benchmark(
    {
        "one_batch_of_six": run_six_as_one_batch,
        "six_separate_one_link_calls": run_six_sequential_single_links,
    },
    warmups=PROFILE_WARMUPS,
    repeats=PROFILE_REPEATS,
)

print()
print("=" * 80)
print("Six-link batch versus six single-link calls")
print("=" * 80)

for name, summary in batch_sequential_summaries.items():
    print_summary(name, summary)

batch_median = batch_sequential_summaries[
    "one_batch_of_six"
]["median_s"]
sequential_median = batch_sequential_summaries[
    "six_separate_one_link_calls"
]["median_s"]

print()
print(
    "sequential change relative to batch:",
    f"{sequential_median - batch_median:+.3f} s "
    f"({100.0 * (sequential_median / batch_median - 1.0):+.1f}%)",
)
print("max_abs:", f"{batch_sequential_comparison['max_abs']:.6e}")
print("rel_l2:", f"{batch_sequential_comparison['rel_l2']:.6e}")

batch_rows = []
for name, summary in batch_sequential_summaries.items():
    row = {"route": name}
    row.update(summary)
    batch_rows.append(row)

save_rows(batch_rows, f"{RESULTS_PREFIX}_batch_vs_sequential.csv")


Round 01/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 02/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 03/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 04/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 05/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 06/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 07/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 08/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 09/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 10/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 11/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 12/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 13/15: order=['one_batch_of_six', 'six_separate_one_link_calls']
Round 14/15: order=['six_separate_one_link_calls', 'one_batch_of_six']
Round 

## Individual-link timings

All six jobs have identical array dimensions and lag count. Large persistent differences would therefore suggest delay-field, data-layout, or geometry-dependent behaviour rather than simple operation count.


In [12]:
# --- 12. Per-link shift timing ---

NUM_LINKS = shift_plan.num_jobs
assert NUM_LINKS == 6

# B=1 was compiled in the scaling section. Warm each link's data path once.
for link_index in range(NUM_LINKS):
    result = run_direct_kernel(link_index, link_index + 1)
    block_result(result)
    del result

link_times = {link_index: [] for link_index in range(NUM_LINKS)}
link_indices = tuple(range(NUM_LINKS))

for repeat_index in range(LINK_REPEATS):
    rotation = repeat_index % NUM_LINKS
    order = list(link_indices[rotation:] + link_indices[:rotation])

    if repeat_index % 2 == 1:
        order = list(reversed(order))

    print(
        f"Link round {repeat_index + 1:02d}/{LINK_REPEATS}: "
        f"order={order}"
    )

    for link_index in order:
        started = time.perf_counter()
        result = run_direct_kernel(link_index, link_index + 1)
        block_result(result)
        link_times[link_index].append(time.perf_counter() - started)
        del result

link_summaries = {
    link_index: summarize_timings(values)
    for link_index, values in link_times.items()
}

link_medians = np.asarray(
    [link_summaries[index]["median_s"] for index in range(NUM_LINKS)]
)

link_spread_percent = (
    100.0
    * (np.max(link_medians) - np.min(link_medians))
    / np.mean(link_medians)
)

link_rows = []

print()
print("=" * 80)
print("Individual one-arm link shift timings")
print("=" * 80)
print(f"{'link':>5} {'median_s':>10} {'IQR_s':>10}")

for link_index in range(NUM_LINKS):
    summary = link_summaries[link_index]
    print(
        f"{link_index:>5d} "
        f"{summary['median_s']:>10.3f} "
        f"{summary['iqr_s']:>10.3f}"
    )

    row = {"link_index": link_index}
    row.update(summary)
    link_rows.append(row)

print()
print("Median link-time spread:", f"{link_spread_percent:.2f}%")

save_rows(link_rows, f"{RESULTS_PREFIX}_individual_links.csv")


Link round 01/9: order=[0, 1, 2, 3, 4, 5]
Link round 02/9: order=[0, 5, 4, 3, 2, 1]
Link round 03/9: order=[2, 3, 4, 5, 0, 1]
Link round 04/9: order=[2, 1, 0, 5, 4, 3]
Link round 05/9: order=[4, 5, 0, 1, 2, 3]
Link round 06/9: order=[4, 3, 2, 1, 0, 5]
Link round 07/9: order=[0, 1, 2, 3, 4, 5]
Link round 08/9: order=[0, 5, 4, 3, 2, 1]
Link round 09/9: order=[2, 3, 4, 5, 0, 1]

Individual one-arm link shift timings
 link   median_s      IQR_s
    0      0.097      0.005
    1      0.096      0.002
    2      0.096      0.003
    3      0.095      0.003
    4      0.099      0.003
    5      0.097      0.003

Median link-time spread: 3.84%


## Static working-set diagnostic

This is not a measured peak-memory profile. It reports the size of the dominant row-by-lag-by-frequency blocks implied by the current schedule. A large block relative to cache is evidence that the kernel may be limited by memory traffic and temporary-array pressure rather than Python overhead.


In [13]:
# --- 13. Static kernel working-set estimates ---

block_elements = ROW_CHUNK_SIZE * LAG_BLOCK_SIZE * wdm.Nf
complex_itemsize = np.dtype(np.complex64).itemsize
real_itemsize = np.dtype(np.float32).itemsize

complex_block_mib = block_elements * complex_itemsize / 1024**2
real_block_mib = block_elements * real_itemsize / 1024**2

print("=" * 80)
print("Static row x lag x frequency working-set estimates")
print("=" * 80)
print("Block shape:", (ROW_CHUNK_SIZE, LAG_BLOCK_SIZE, wdm.Nf))
print("Elements:", f"{block_elements:,}")
print("One complex64 block:", f"{complex_block_mib:.2f} MiB")
print("One float32 block:", f"{real_block_mib:.2f} MiB")
print(
    "Three simultaneous complex64-sized blocks:",
    f"{3.0 * complex_block_mib:.2f} MiB",
)
print(
    "Six-link output:",
    f"{shift_plan.num_jobs * wdm.Nt * wdm.Nf * complex_itemsize / 1024**2:.2f} MiB",
)
print(
    "Persistent device plan:",
    f"{shift_plan.device_plan_memory_bytes / 1024**2:.2f} MiB",
)
print("Current process RSS:", f"{rss_gib():.2f} GiB")


Static row x lag x frequency working-set estimates
Block shape: (4096, 32, 100)
Elements: 13,107,200
One complex64 block: 100.00 MiB
One float32 block: 50.00 MiB
Three simultaneous complex64-sized blocks: 300.00 MiB
Six-link output: 149.41 MiB
Persistent device plan: 331.20 MiB
Current process RSS: 10.30 GiB


## Optional JAX profiler trace

Set `RUN_JAX_TRACE=True` in the configuration cell and rerun this cell. It records several warmed production-shift applications without changing the source code.

The trace directory can be opened with TensorBoard, for example:

```bash
tensorboard --logdir jax_profile_one_arm_shift
```

Depending on the installed JAX build, CPU trace detail may vary.


In [14]:
# --- 14. Optional JAX trace ---

if RUN_JAX_TRACE:
    TRACE_DIRECTORY.mkdir(parents=True, exist_ok=True)

    # Ensure compilation and device-plan population are outside the trace.
    warm = run_shift_only()
    block_result(warm)
    del warm

    try:
        with jax.profiler.trace(
            str(TRACE_DIRECTORY),
            create_perfetto_link=False,
        ):
            for _ in range(TRACE_REPEATS):
                result = run_shift_only()
                block_result(result)
                del result

        print("JAX trace written to:", TRACE_DIRECTORY.resolve())
        print(
            "Open with: tensorboard --logdir",
            TRACE_DIRECTORY,
        )
    except Exception as exc:
        print("JAX trace failed:", repr(exc))
        print(
            "The numerical timing sections remain valid; "
            "trace support depends on the local JAX installation."
        )
else:
    print("JAX trace skipped. Set RUN_JAX_TRACE=True to enable it.")


JAX trace skipped. Set RUN_JAX_TRACE=True to enable it.


## Interpretation guide

Use the results in this order:

1. **Shift share of complete A1**
   - If still around 95–98%, modulation/materialization and host-device conversion are not worthwhile targets.

2. **Wrapper versus direct kernel**
   - Near-zero difference means the remaining cost is inside the compiled numerical kernel.

3. **Batch scaling**
   - \(R^2\) close to one and nearly constant seconds per link indicate repeated per-link arithmetic is dominant.
   - A large intercept indicates shared setup or dispatch overhead.

4. **Batch versus six one-link calls**
   - Similar timings mean batching is not the problem.
   - A substantially slower six-link batch points to the batch `fori_loop` or compiled layout.

5. **Individual links**
   - Similar link timings rule out one anomalous delay field.
   - A persistent outlier should be investigated against its delay statistics and link geometry.

6. **Static working set**
   - Large row-lag-frequency blocks support a memory-traffic or temporary-array-pressure hypothesis.

If the wrapper is negligible, scaling is linear, the six links are similar, and batching is not worse than sequential execution, the next source-level profiling step should split the compiled kernel into separately timed carrier and sideband kernels. That requires an explicitly diagnostic backend; it should not be inferred from Python profiling alone.


## Experimental shared-mask reuse

This isolated experiment compares the production lag-blocked shift with a candidate that:

- masks `w_n_block` once per row/lag/frequency block;
- reuses the masked waveform block in the carrier and both sidebands;
- computes `conj(Cnm[target_rows])` once and reuses its frequency slices.

No production dispatcher or plan code is changed.


### Rejected shared-mask experiment

At `row_chunk_size=4096`, `lag_block_size=32`, complex64:

- baseline median: 0.650 s
- candidate median: 0.649 s
- runtime change: -0.02%
- IQRs overlapping
- relative L2 difference: 3.99e-08

Decision: retain production baseline.